# 03. Обнаружение точек структурных изменений (шоков)

**Задача** — заметить шок как можно раньше и не поднимать ложных тревог. Поэтому все методы работают
**в онлайн-режиме**: в каждом месяце `t` детектор видит только данные до `t` включительно и решает,
поднимать ли тревогу. Так измеряется реальная задержка обнаружения, а не точность «задним числом».

**Методы.**
- `PELT` и `BinSeg` (библиотека ruptures) — классическая сегментация; в онлайн-режиме перезапускаются
  на расширяющемся окне, тревога — если найдена новая точка изменения в последних 3 месяцах;
- `CUSUM` — накопленная сумма отклонений от исторического уровня (двусторонняя);
- `BOCPD` — байесовское онлайн-обнаружение (Adams & MacKay, 2007): вероятность, что «сейчас начался новый режим»;
- `EWMA-остатки` — прогнозный детектор: отклонение факта от прогноза по недавнему уровню в единицах σ.

**Проверка на двух наборах.**
1. **Синтетика на реальных рядах МО** — в половину рядов вставляем сдвиг уровня известного размера в известный
   месяц, во вторую половину — нет (для подсчёта ложных тревог). Пороги подбираются на калибровочной половине МО,
   метрики считаются на другой половине.
2. **Национальные ряды СберИндекса 2018–2026** с реальными шоками: COVID (апрель 2020) и март 2022.

**Метрики.** Precision, Recall, F1 (тревога засчитывается, если пришла не позже чем через 2 месяца
после шока и указывает на него с точностью ±1 месяц), средняя задержка обнаружения, доля ложных тревог.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import ruptures as rpt
import yaml
from scipy import stats
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CFG = yaml.safe_load(open(ROOT / "configs" / "changepoints.yaml", encoding="utf-8"))
OUT = ROOT / "results"; OUT.mkdir(exist_ok=True)
RAW = ROOT / "data" / "raw"
rng = np.random.default_rng(CFG["seed"])
TOL, MAXDELAY, WARMUP = CFG["eval"]["loc_tolerance"], CFG["eval"]["max_delay"], CFG["eval"]["warmup"]

## 1. Детекторы (единый онлайн-интерфейс: ряд → список тревог `(месяц тревоги, оценка месяца шока)`)

In [ ]:
def _z(x):
    base = x[:WARMUP]; s = base.std() or 1.0
    return (x - base.mean()) / s

def online_segmenter(x, algo, pen):
    alarms, last = [], -99
    x = _z(x)                                                    # штраф в единицах σ — сравним между рядами
    for t in range(WARMUP, len(x)):
        seg = x[: t + 1]
        model = (rpt.Pelt(model="l2", min_size=2, jump=1) if algo == "pelt"
                 else rpt.Binseg(model="l2", min_size=2, jump=1)).fit(seg.reshape(-1, 1))
        cps = [c for c in model.predict(pen=pen) if c < len(seg)]
        new = [c for c in cps if c >= t - 2 and abs(c - last) > 2]
        if new:
            last = new[-1]; alarms.append((t, last))
    return alarms

def cusum(x, h, k=0.5):
    z, alarms, sp, sn, start = _z(x), [], 0.0, 0.0, 0
    for t in range(WARMUP, len(x)):
        sp, sn = max(0, sp + z[t] - k), max(0, sn - z[t] - k)
        if sp > h or sn > h:
            alarms.append((t, max(start, t - 1)))
            sp = sn = 0.0; start = t
            z = z.copy(); z[t:] = z[t:] - z[t]                 # новый уровень становится базовым
    return alarms

def bocpd(x, p_thr, hazard=1 / 12):
    """Adams & MacKay: модель Normal-Gamma, предсказательное распределение Стьюдента."""
    z = _z(x); T = len(z)
    mu0, k0, a0, b0 = 0.0, 1.0, 1.0, 1.0
    R = np.array([1.0]); mu, kap, al, be = np.array([mu0]), np.array([k0]), np.array([a0]), np.array([b0])
    alarms, last = [], -99
    for t in range(T):
        scale = np.sqrt(be * (kap + 1) / (al * kap))
        pred = stats.t.pdf(z[t], 2 * al, mu, scale)
        growth = R * pred * (1 - hazard); cp = (R * pred * hazard).sum()
        R = np.append(cp, growth); R /= R.sum()
        mu_n = (kap * mu + z[t]) / (kap + 1)
        be = np.append(b0, be + kap * (z[t] - mu) ** 2 / (2 * (kap + 1)))
        mu, kap, al = np.append(mu0, mu_n), np.append(k0, kap + 1), np.append(a0, al + 0.5)
        if t >= WARMUP:
            p_recent = R[1:3].sum()                              # новый режим начался 1–2 месяца назад
            if p_recent > p_thr and t - last > 2:
                last = t; alarms.append((t, t - int(np.argmax(R[1:3]))))
    return alarms

def ewma_resid(x, thr, win=6):
    """Прогноз = средний уровень за прошлые `win` месяцев; тревога, если факт отклонился больше чем на thr·σ."""
    alarms, last = [], -99
    for t in range(WARMUP, len(x)):
        hist = x[max(0, t - win): t]; s = hist.std(ddof=1) if len(hist) > 2 else np.nan
        if not s or np.isnan(s): continue
        if abs(x[t] - hist.mean()) / s > thr and t - last > 2:
            last = t; alarms.append((t, t))
    return alarms

DETECTORS = {
    "PELT": lambda x, p: online_segmenter(x, "pelt", p),
    "BinSeg": lambda x, p: online_segmenter(x, "binseg", p),
    "CUSUM": lambda x, p: cusum(x, p),
    "BOCPD": lambda x, p: bocpd(x, p),
    "EWMA-остатки": lambda x, p: ewma_resid(x, p),
}
GRID = CFG["grid"]

## 2. Синтетика на реальных рядах МО
Берём относительный уровень МО: `log(расходы МО) − медиана log(расходов всех МО)` в том же месяце.
Это убирает общую сезонность (декабрьский пик) и общероссийский тренд — остаётся собственная динамика МО.
В половину рядов вставляем сдвиг уровня от −25% до −8% или от +8% до +25% в случайный месяц.

In [ ]:
panel = pd.read_parquet(ROOT / CFG["data"]["panel"])
y = panel[panel.date >= "2023-01-01"].pivot(index="mo_id", columns="date", values="y_all")
y = y.dropna()                                                  # только полные 24 месяца
mos = y.index.to_numpy(); rng.shuffle(mos)
mos = mos[: CFG["synthetic"]["n_series"]]
Y = np.log(y.loc[mos].to_numpy())
T = Y.shape[1]

shock_t = np.full(len(mos), -1)
has = rng.random(len(mos)) < CFG["synthetic"]["share_with_shock"]
lo, hi = CFG["synthetic"]["shift_range"]
for i in np.where(has)[0]:
    tau = rng.integers(WARMUP + 1, T - 2)
    size = rng.uniform(lo, hi) * rng.choice([-1, 1])
    Y[i, tau:] += np.log1p(size); shock_t[i] = tau
R = Y - np.median(Y, axis=0)                                    # относительный уровень
calib = np.arange(len(mos)) < len(mos) // 2

def evaluate(alarms_list, truth):
    tp = fp = fn = 0; delays = []
    for alarms, tau in zip(alarms_list, truth):
        hit = False
        for t_al, loc in alarms:
            ok = tau >= 0 and tau <= t_al <= tau + MAXDELAY and abs(loc - tau) <= TOL and not hit
            if ok: hit = True; delays.append(t_al - tau)
            else: fp += 1
        if tau >= 0: tp += hit; fn += (not hit)
    prec = tp / (tp + fp) if tp + fp else 0.0; rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    no_shock = [a for a, t in zip(alarms_list, truth) if t < 0]
    far = np.mean([len(a) > 0 for a in no_shock]) if no_shock else np.nan
    return {"precision": prec, "recall": rec, "F1": f1, "задержка_мес": np.mean(delays) if delays else np.nan,
            "доля_рядов_без_шока_с_ложной_тревогой": far}

rows, best = [], {}
for name, det in DETECTORS.items():
    scores = []
    for p in GRID[name]:                                        # подбор порога только на калибровочной половине
        res = evaluate([det(R[i], p) for i in np.where(calib)[0]], shock_t[calib])
        scores.append((res["F1"], p))
    best[name] = max(scores)[1]
    res = evaluate([det(R[i], best[name]) for i in np.where(~calib)[0]], shock_t[~calib])
    rows.append({"метод": name, "параметр": best[name], **res})
    print(f"{name}: параметр {best[name]}, F1={res['F1']:.3f}")
synth = pd.DataFrame(rows).sort_values("F1", ascending=False)
synth.to_csv(OUT / "changepoint_synthetic.csv", index=False)
print(synth.round(3).to_string(index=False))

### Чувствительность к размеру шока
Насколько хорошо лучший метод ловит маленькие и большие сдвиги.

In [ ]:
sizes = np.abs(np.exp(Y[:, -1] - np.log(y.loc[mos].to_numpy())[:, -1]) - 1)
test_idx = np.where(~calib & has)[0]
bins = pd.cut(sizes[test_idx], [0, 0.12, 0.18, 0.26], labels=["8–12%", "12–18%", "18–25%"])
sens = []
for name in synth["метод"]:
    hits = []
    for i in test_idx:
        al = DETECTORS[name](R[i], best[name])
        hits.append(any(shock_t[i] <= t <= shock_t[i] + MAXDELAY and abs(l - shock_t[i]) <= TOL for t, l in al))
    sens.append(pd.Series(hits).groupby(bins).mean().rename(name))
sens = pd.concat(sens, axis=1).T.round(2); sens.to_csv(OUT / "changepoint_recall_by_size.csv")
print("Recall по размеру шока:\n", sens.to_string())

## 3. Реальные шоки: национальные ряды СберИндекса 2018–2026
Ряды с сезонной корректировкой (индекс СберИндекса, дек. 2018 = 100) по пяти категориям трат.
Окна событий: COVID — март–август 2020 (обвал и восстановление), шок 2022 года — март–май 2022.
Пороги берём те же, что подобраны на синтетике, — никакой подгонки под реальные события.

In [ ]:
sa = pd.read_parquet(RAW / CFG["data"]["sber_sa"]); sa["date"] = pd.to_datetime(sa.period)
nat = sa.pivot(index="date", columns="type", values="value").sort_index()
EVENTS = {k: (pd.Timestamp(v[0]), pd.Timestamp(v[1]), pd.Timestamp(v[2])) for k, v in CFG["events"].items()}

real = []
for name, det in DETECTORS.items():
    for col in nat.columns:
        x = np.log(nat[col].to_numpy()); dates = nat.index
        x = np.r_[0.0, np.diff(x)] if name == "EWMA-остатки" else x   # для прогнозного детектора — темпы роста
        al = [(dates[t], dates[l]) for t, l in det(x, best[name])]
        for ev, (start, end, shock) in EVENTS.items():
            in_ev = [a for a in al if start <= a[1] <= end and a[0] <= shock + pd.DateOffset(months=MAXDELAY)]
            real.append({"метод": name, "ряд": col, "событие": ev, "найдено": bool(in_ev),
                         "задержка_мес": ((in_ev[0][0].year - shock.year) * 12 + in_ev[0][0].month - shock.month) if in_ev else np.nan})
        outside = [a for a in al if not any(s <= a[1] <= e for s, e, _ in EVENTS.values())]
        real.append({"метод": name, "ряд": col, "событие": "вне событий (ложные)", "найдено": len(outside), "задержка_мес": np.nan})
real = pd.DataFrame(real)
summary = (real[real.событие != "вне событий (ложные)"].groupby(["метод", "событие"])
           .agg(найдено_рядов=("найдено", "sum"), задержка_мес=("задержка_мес", "mean")).unstack())
summary.columns = [f"{a}_{b}" for a, b in summary.columns]
false = real[real.событие == "вне событий (ложные)"].groupby("метод").найдено.sum().rename("ложных_тревог_2019_2026")
summary = summary.join(false); summary.to_csv(OUT / "changepoint_real_events.csv")
print(f"Из {nat.shape[1]} рядов найдено:\n", summary.round(1).to_string())

## 4. Иллюстрация для презентации

In [ ]:
top = synth.iloc[0]["метод"]
fig, ax = plt.subplots(figsize=(11, 4))
col = "Всего"; ax.plot(nat.index, nat[col], color="#333", lw=1.8, label=f"СберИндекс, {col.lower()} (с.к.)")
x = np.log(nat[col].to_numpy()); x = np.r_[0.0, np.diff(x)] if top == "EWMA-остатки" else x
for t, l in DETECTORS[top](x, best[top]):
    ax.axvline(nat.index[t], color="#d62728", ls="--", lw=1)
for ev, (s, e, _) in EVENTS.items():
    ax.axvspan(s, e, color="#ff7f0e", alpha=0.12)
ax.set_title(f"Тревоги детектора {top} на реальных данных (красные линии), окна известных шоков — оранжевым")
ax.legend(loc="lower right"); fig.tight_layout(); fig.savefig(OUT / "changepoints_national.png", dpi=150)